# Pasado `gpu-hybrid`: TorchInductor, prag za automatski izbor solvera i A3 end-to-end

Pre pokretanja: **Runtime > Change runtime type > GPU (A100)**, zatim **Runtime > Disconnect and delete runtime**, pa **Runtime > Run all**.

Prethodna sveska (`results/planar_fit_cuda_2026-10-27/`) je pokazala da A2 (`planar_fit.fit_planes`) košta oko 3.8 ms po pozivu bez obzira na broj boksova, pa je na malim mrežama sa `n_splits = 1` (100 boksova po pozivu) sporiji od `gelsd`. Lokalno: Dynamo hvata `fit_planes` kao jedan graf bez prekida sa 292 operacije, što odgovara launch overhead-u.

Tri eksperimenta, merena jedan za drugim (ne paralelno) na istom GPU-u:

1. **TorchInductor na `fit_planes`**: eager, `torch.compile(fullgraph=True)`, isto sa `mode="reduce-overhead"` (CUDA graph), i sa `dynamic=True`. Za svaki oblik: vreme prvog poziva (kompilacija), steady-state vreme, broj CUDA kernela po pozivu, broj grafova i prekida, odstupanje koeficijenata od eager A2.
2. **Prag (crossover)**: G, A2 eager, najbolji kompajlirani A2 i A3 za `n = 64 ... 65536`, `float64` i `float32`. Prag = najmanji `n` na kome je metod bar 20% brži od G i ostaje brži na svim većim `n`.
3. **End-to-end**: sekvencijalni G, batched G, batched A2 eager, batched A2 kompajliran, batched A3; 4 mreže, `eps_idx` 0 i 8, `n_splits` 1, 8, 32, 64, 3 slike. Plus simulirani automatski izbor po pragu.

Ništa u kodu se ne menja; ovo je merenje. Očekivano trajanje: oko 40 do 60 minuta (procena; kompilacija za svaki od oko 70 oblika traje po nekoliko sekundi do desetak). Kad završi, pošalji `inductor_dispatch_a3.zip`.

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Uključi GPU: Runtime > Change runtime type > GPU"
GPU = torch.cuda.get_device_name(0)
VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| GPU:", GPU,
      "| VRAM GiB:", round(VRAM_GIB, 1))
REQUIRE_A100 = True
assert "A100" in GPU or not REQUIRE_A100, f"Merenje je dogovoreno na A100, a ovo je {GPU}."

In [ ]:
import os
import shutil

REPO = "/content/Pasado"
URL = "https://github.com/dimitrijepesic/Pasado.git"
COMMIT = "7e4cdcd"  # planar_fit behind the switch, candidates incl. A3, harness

os.chdir("/content")
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "-c", "http.version=HTTP/1.1", "clone", "--branch", "gpu-hybrid",
                "--single-branch", URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", COMMIT], check=True)
head = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert head.startswith(COMMIT), f"HEAD je {head}, a ocekivan je {COMMIT}"
print("HEAD:", head)
os.chdir(REPO)

OUT = "/content/inductor_dispatch_a3"
os.makedirs(OUT, exist_ok=True)

In [ ]:
%pip install -q scikit-learn mpmath

In [ ]:
import json
import math
import statistics
import sys
import time

import pandas as pd
from IPython.display import display

for p in (f"{REPO}/forward_mode_tensorized_src", f"{REPO}/Section_5_4", f"{REPO}/experiments"):
    if p not in sys.path:
        sys.path.insert(0, p)

import torch._dynamo
import batched_splits as bs
import lstsq_candidates as lc
import planar_fit
import precise_transformer as pt
from compare_lstsq_solvers import (EPS_SCHEDULE, grid_and_targets, set_production_flags,
                                   swapped_solver, synthetic_bounds)
from measure_split_sweep import build_forward_precise, eval_splits_once, load_network_and_images

set_production_flags()
torch.set_default_dtype(torch.float64)
device = torch.device("cuda")

# one compiled function sees many shapes; the default limit of 8 would silently fall
# back to eager after the eighth
for name in ("cache_size_limit", "recompile_limit"):
    if hasattr(torch._dynamo.config, name):
        setattr(torch._dynamo.config, name, 256)

N_LIST = [64, 100, 128, 200, 256, 400, 512, 800, 1024, 2048, 8192, 65536]
DTYPES = ["float64", "float32"]
WARMUP, REPEATS = 3, 20


def boxes(n, dtype_name, seed=11):
    b = synthetic_bounds("normal", n, seed, dtype_name)
    return grid_and_targets(*(t.to(device=device, dtype=getattr(torch, dtype_name)) for t in b))


def first_call_seconds(fn, *args):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    fn(*args)
    torch.cuda.synchronize()
    return time.perf_counter() - t0


def steady_ms(fn, *args, warmup=WARMUP, repeats=REPEATS):
    for _ in range(warmup):
        fn(*args)
    times = []
    for _ in range(repeats):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn(*args)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000.0)
    return statistics.median(times)


def kernels_per_call(fn, *args):
    """CUDA kernels launched by one call (None if the profiler does not report them)."""
    try:
        from torch.autograd import DeviceType
        from torch.profiler import ProfilerActivity, profile
        torch.cuda.synchronize()
        with profile(activities=[ProfilerActivity.CUDA]) as prof:
            fn(*args)
            torch.cuda.synchronize()
        names = [e.name for e in prof.events() if e.device_type == DeviceType.CUDA]
        return len([n for n in names if not n.lower().startswith("memcpy")
                    and not n.lower().startswith("memset")])
    except Exception as e:
        print("profiler:", type(e).__name__, e)
        return None


def rel_diff(a, b):
    scale = b.abs().max(dim=1).values.clamp_min(torch.finfo(b.dtype).tiny)
    return ((a - b).abs().max(dim=1).values / scale).max().item()

## 1. TorchInductor na `fit_planes`

Za svaku varijantu nova kompilacija (`torch._dynamo.reset()`), zatim svi oblici redom. `first_call_s` uključuje kompilaciju (ili rekompilaciju) za taj oblik; `steady_ms` je medijana 20 poziva posle 3 warm-up-a. Kompajlirani A2 nije bit-identičan eager-u (fuzija menja redosled zaokruživanja), pa se beleži `max_rel_diff_vs_eager`.

In [ ]:
with torch.no_grad():
    for dtype_name in DTYPES:
        g, z = boxes(1024, dtype_name)
        ex = torch._dynamo.explain(planar_fit.fit_planes)(g, z)
        print(f"{dtype_name}: graphs {ex.graph_count}, graph breaks {ex.graph_break_count}, "
              f"ops per graph {[len(o) for o in ex.ops_per_graph]}")
        for r in ex.break_reasons:
            print("   break:", r.reason)
torch._dynamo.reset()

VARIANTS = {
    "eager": None,
    "compile": dict(fullgraph=True),
    "compile_reduce_overhead": dict(fullgraph=True, mode="reduce-overhead"),
    "compile_dynamic": dict(fullgraph=True, dynamic=True),
}

inductor_rows = []
with torch.no_grad():
    for variant, opts in VARIANTS.items():
        torch._dynamo.reset()
        fn = planar_fit.fit_planes if opts is None else torch.compile(planar_fit.fit_planes, **opts)
        for dtype_name in DTYPES:
            for n in N_LIST:
                g, z = boxes(n, dtype_name)
                row = dict(variant=variant, dtype=dtype_name, n=n)
                try:
                    row["first_call_s"] = first_call_seconds(fn, g, z)
                    row["steady_ms"] = steady_ms(fn, g, z)
                    row["kernels_per_call"] = kernels_per_call(fn, g, z)
                    out = fn(g, z).clone()
                    ref = planar_fit.fit_planes(g, z)
                    row["max_rel_diff_vs_eager"] = rel_diff(out, ref)
                    row["bitwise_vs_eager"] = bool(torch.equal(out, ref))
                    row["status"] = "ok"
                except Exception as e:
                    row["status"] = f"error: {type(e).__name__}: {str(e).splitlines()[0][:150]}"
                inductor_rows.append(row)
                print(variant, dtype_name, n, row.get("steady_ms"), row["status"], flush=True)
        try:
            row_stats = dict(torch._dynamo.utils.counters["stats"])
            print(variant, "dynamo stats:", row_stats)
        except Exception:
            pass

inductor = pd.DataFrame(inductor_rows)
inductor.to_csv(f"{OUT}/inductor_fit_planes.csv", index=False)
for dtype_name in DTYPES:
    print(dtype_name, "steady ms")
    display(inductor[inductor.dtype == dtype_name].pivot(index="variant", columns="n", values="steady_ms").round(3))
    print(dtype_name, "kernels per call")
    display(inductor[inductor.dtype == dtype_name].pivot(index="variant", columns="n", values="kernels_per_call"))
    print(dtype_name, "first call s (includes compilation)")
    display(inductor[inductor.dtype == dtype_name].pivot(index="variant", columns="n", values="first_call_s").round(2))

Izbor najboljeg kompajliranog A2 za naredne korake: varijanta koja radi na svim oblicima u `float64`, odstupa od eager-a najviše `1e-10`, i ima najmanji zbir steady-state vremena na `n = 100, 1024, 65536`.

In [ ]:
ok = inductor[(inductor.variant != "eager") & (inductor.dtype == "float64")]
good = []
for variant, sub in ok.groupby("variant"):
    if (sub.status == "ok").all() and (sub.max_rel_diff_vs_eager <= 1e-10).all():
        good.append((sub[sub.n.isin([100, 1024, 65536])].steady_ms.sum(), variant))
BEST = min(good)[1] if good else None
print("best compiled variant:", BEST)

## 2. Prag: G, A2 eager, A2 kompajliran, A3

G uključuje prenos na CPU i nazad (kao u produkciji). Prag za metod je najmanji `n` na kome je bar 20% brži od G (`t_G / t >= 1.2`) i ostaje brži (`>= 1.0`) na svim većim `n`.

In [ ]:
cross_rows = []
with torch.no_grad():
    torch._dynamo.reset()
    compiled = torch.compile(planar_fit.fit_planes, **VARIANTS[BEST]) if BEST else None
    methods = {"G": lc.solve_G, "A2_eager": planar_fit.fit_planes, "A3": lc.solve_A3}
    if compiled is not None:
        methods[f"A2_{BEST}"] = compiled
    for dtype_name in DTYPES:
        for n in N_LIST:
            g, z = boxes(n, dtype_name)
            row = dict(dtype=dtype_name, n=n)
            for name, fn in methods.items():
                row[f"ms_{name}"] = steady_ms(fn, g, z)
            cross_rows.append(row)
            print(dtype_name, n, {k: round(v, 3) for k, v in row.items() if k.startswith("ms_")}, flush=True)

cross = pd.DataFrame(cross_rows)
names = [c[3:] for c in cross.columns if c.startswith("ms_") and c != "ms_G"]
for name in names:
    cross[f"speedup_{name}_vs_G"] = cross.ms_G / cross[f"ms_{name}"]
cross.to_csv(f"{OUT}/crossover.csv", index=False)


def threshold(sub, name):
    s = sub.sort_values("n")
    ns, r = list(s.n), list(s[f"speedup_{name}_vs_G"])
    for i, n in enumerate(ns):
        if r[i] >= 1.2 and all(x >= 1.0 for x in r[i:]):
            return int(n)
    return None


thresholds = {dt: {name: threshold(cross[cross.dtype == dt], name) for name in names} for dt in DTYPES}
print(json.dumps(thresholds, indent=2))
with open(f"{OUT}/thresholds.json", "w") as f:
    json.dump(dict(thresholds=thresholds, best_compiled=BEST, rule="t_G/t >= 1.2 at n and >= 1.0 above"), f, indent=2)
for dt in DTYPES:
    print(dt)
    display(cross[cross.dtype == dt][["n"] + [c for c in cross.columns if c.startswith("speedup_")]].round(2))

## 3. End-to-end: sekvencijalni G, batched G, A2 eager, A2 kompajliran, A3

Sve u istom runu. Pozivi solvera se broje u svakom režimu (nula zatvorenih formi u G režimima). Granice moraju da se slažu sa sekvencijalnom petljom do `1e-9`. Prvi prolaz za svaku konfiguraciju je warm-up (uključuje kompilaciju za nove oblike) i beleži se posebno.

In [ ]:
SPLITS = [1, 8, 32, 64]
EPS_INDICES = [0, 8]
NUM_IMAGES = 3
E2E_REPEATS = 3
BIG_CHUNK = min(64, 2 ** int(math.log2(max(1, 0.6 * VRAM_GIB / 0.5))))

torch._dynamo.reset()
compiled = torch.compile(planar_fit.fit_planes, **VARIANTS[BEST]) if BEST else None
calls = {"closed": 0, "swapped": 0}
fit_planes_eager = planar_fit.fit_planes


def counted_eager(grid, z, rcond=None):
    calls["closed"] += 1
    return fit_planes_eager(grid, z, rcond)


planar_fit.fit_planes = counted_eager


def swapped(fn, clone):
    def run(grid, z):
        calls["swapped"] += 1
        out = fn(grid, z)
        return out.clone() if clone else out
    return run


MODES = ["sequential_G", "batched_G", "batched_A2_eager", "batched_A3"]
if compiled is not None:
    MODES.insert(3, "batched_A2_compiled")
clone_out = BEST == "compile_reduce_overhead"  # CUDA graph outputs are reused by the next replay


def run_mode(mode, layers, forward_fn, img, eps, n_splits, chunk):
    bs.USE_CLOSED_FORM_FIT = mode == "batched_A2_eager"
    if mode == "sequential_G":
        return lambda: eval_splits_once(forward_fn, img, eps, n_splits, 7, device, torch.float64), None
    if mode in ("batched_G", "batched_A2_eager"):
        return lambda: bs.lipschitz_bound(layers, img, eps, n_splits, seed=7, chunk=chunk), None
    fn = swapped(compiled, clone_out) if mode == "batched_A2_compiled" else swapped(lc.solve_A3, False)
    return lambda: bs.lipschitz_bound(layers, img, eps, n_splits, seed=7, chunk=chunk), fn


e2e_rows = []
with torch.no_grad():
    for network in ["3layer", "4layer", "5layer", "big"]:
        net, images = load_network_and_images(network, device, torch.float64, NUM_IMAGES)
        forward_fn = build_forward_precise(pt, net, network)
        layers = bs.extract_affine_layers(net, network)
        width = 1024 if network == "big" else 100
        chunk = BIG_CHUNK if network == "big" else None
        for image_id, img in images:
            for eps_idx in EPS_INDICES:
                eps = EPS_SCHEDULE[eps_idx]
                for n_splits in SPLITS:
                    row = dict(network=network, image_id=image_id, eps_idx=eps_idx, n_splits=n_splits,
                               boxes_per_call=n_splits * width)
                    for mode in MODES:
                        call, solver = run_mode(mode, layers, forward_fn, img, eps, n_splits, chunk)
                        calls["closed"] = calls["swapped"] = 0
                        ctx = swapped_solver(solver) if solver is not None else None
                        if ctx is not None:
                            ctx.__enter__()
                        try:
                            torch.cuda.synchronize()
                            t0 = time.perf_counter()
                            call()  # warm-up, compiles new shapes
                            torch.cuda.synchronize()
                            row[f"first_ms_{mode}"] = (time.perf_counter() - t0) * 1000.0
                            times = []
                            for _ in range(E2E_REPEATS):
                                torch.cuda.synchronize()
                                t0 = time.perf_counter()
                                bound = call()
                                torch.cuda.synchronize()
                                times.append((time.perf_counter() - t0) * 1000.0)
                        finally:
                            if ctx is not None:
                                ctx.__exit__(None, None, None)
                            bs.USE_CLOSED_FORM_FIT = False
                        expect_closed = mode == "batched_A2_eager"
                        expect_swapped = mode in ("batched_A2_compiled", "batched_A3")
                        assert (calls["closed"] > 0) == expect_closed, (mode, calls)
                        assert (calls["swapped"] > 0) == expect_swapped, (mode, calls)
                        row[f"ms_{mode}"] = statistics.median(times)
                        row[f"bound_{mode}"] = bound
                    e2e_rows.append(row)
                print(f"{network} image {image_id} eps_idx {eps_idx} done", flush=True)
planar_fit.fit_planes = fit_planes_eager

e2e = pd.DataFrame(e2e_rows)
for mode in MODES[1:]:
    e2e[f"rel_diff_{mode}"] = (e2e[f"bound_{mode}"] - e2e.bound_sequential_G).abs() / e2e.bound_sequential_G.abs()
worst = e2e[[f"rel_diff_{m}" for m in MODES[1:]]].max()
print("largest relative difference to the sequential bound:")
print(worst.to_string())
e2e.to_csv(f"{OUT}/e2e.csv", index=False)
assert (worst <= 1e-9).all(), "granica se ne slaže sa sekvencijalnom; pošalji e2e.csv"

In [ ]:
agg = {f"ms_{m}": (f"ms_{m}", "mean") for m in MODES}
summary = e2e.groupby(["network", "eps_idx", "n_splits", "boxes_per_call"], as_index=False).agg(**agg)
for m in MODES[2:]:
    summary[f"{m}_vs_batched_G"] = summary.ms_batched_G / summary[f"ms_{m}"]
    summary[f"{m}_vs_sequential"] = summary.ms_sequential_G / summary[f"ms_{m}"]

# automatic dispatch simulated with the float64 thresholds of section 2
for name, mode in (("A2_eager", "batched_A2_eager"), (f"A2_{BEST}", "batched_A2_compiled")):
    thr = thresholds["float64"].get(name)
    if mode not in MODES or thr is None:
        continue
    t = summary.apply(lambda r: r.ms_batched_G if r.boxes_per_call < thr else r[f"ms_{mode}"], axis=1)
    summary[f"dispatch_{name}_vs_batched_G"] = summary.ms_batched_G / t
    print(f"dispatch with {name}: gelsd below n = {thr}, closed form from there on")

summary.to_csv(f"{OUT}/e2e_summary.csv", index=False)
pd.set_option("display.width", 250)
display(summary.round(2))

## 4. Ishod (informativno)

- Ako kompajlirani A2 već na 100 boksova po pozivu (male mreže, `n_splits = 1`) nije sporiji od batched G: nema potrebe za pragom, A2 može biti univerzalni default (kompajliran).
- Ako ne: automatski izbor po pragu iz sekcije 2.
- Ako samo A3 rešava male mreže: poseban sigmoidni put, A2 ostaje opšti solver.

In [ ]:
small1 = summary[(summary.network != "big") & (summary.n_splits == 1)]
for m in MODES[2:]:
    r = small1[f"{m}_vs_batched_G"]
    print(f"{m:22s} vs batched G at 100 boxes per call: min {r.min():.2f}x, max {r.max():.2f}x")
for col in [c for c in summary.columns if c.startswith("dispatch_")]:
    print(f"{col}: worst {summary[col].min():.2f}x over all networks and splits")
big = summary[(summary.network == "big") & (summary.n_splits.isin([32, 64]))]
for m in MODES[2:]:
    print(f"{m:22s} big at 32/64 splits vs sequential: {big[f'{m}_vs_sequential'].min():.1f}x to "
          f"{big[f'{m}_vs_sequential'].max():.1f}x")

## 5. Preuzimanje rezultata (`.zip`)

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile
from google.colab import files

with open(f"{OUT}/colab_environment.json", "w") as f:
    json.dump(dict(gpu=GPU, vram_gib=VRAM_GIB, torch=torch.__version__, cuda=torch.version.cuda,
                   commit=head, best_compiled=BEST, big_chunk=BIG_CHUNK, n_list=N_LIST,
                   repeats=REPEATS, e2e_repeats=E2E_REPEATS), f, indent=2)

archive = "/content/inductor_dispatch_a3.zip"
with ZipFile(archive, "w", ZIP_DEFLATED) as zf:
    for name in sorted(os.listdir(OUT)):
        zf.write(os.path.join(OUT, name), os.path.join("inductor_dispatch_a3", name))
print(archive, round(os.path.getsize(archive) / 2**20, 2), "MiB")
files.download(archive)